# Lesson 4.4 — Diagnosing System Issues

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Distinguish prompt failure, hallucination, and model mismatch
- Use signals to localize the root cause quickly
- Encode a diagnosis decision tree for on-call use

**You will produce:** A `diagnose()` decision tree that maps symptoms to likely root causes.

## Concepts

When a Claude system misbehaves, architects triage fast by **symptom -> likely cause**:

- **Prompt failure** — wrong format, ignored instructions, inconsistent output. Usually a prompt/contract problem; fix with clearer instructions, examples, or output schema.
- **Hallucination** — confident, fluent, but unsupported claims. Often a *grounding/retrieval* problem (bad or missing context), not the model being 'dumb'. Sample 3 lives here: wrong answers after a document refresh point to retrieval/indexing.
- **Model mismatch** — task needs more (or less) reasoning than the chosen tier provides; fix by right-sizing (Lesson 2.1).

The discriminating signals: did output *format* break (prompt), did a *document/index* change (retrieval), is the task *reasoning-heavy* and the model small (mismatch)? A decision tree keeps triage disciplined under pressure.

> **Reference:** [Anthropic — Reduce hallucinations](https://docs.anthropic.com/en/docs/test-and-evaluate/strengthen-guardrails/reduce-hallucinations)

In [ ]:
# !pip install anthropic
import os, json, time, random

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        # Deterministic mock: echo a label when the prompt asks for classification
        return f"[MOCK {model}] {prompt[:60]}"
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Encode the diagnosis decision tree

We turn the triage logic into a function over observable signals so on-call engineers get a consistent first hypothesis.

In [ ]:
def diagnose(signals):
    """signals: dict of booleans about the failure."""
    if signals.get("format_broken") or signals.get("instructions_ignored"):
        return "prompt failure", "Tighten instructions/output contract; add few-shot examples"
    if signals.get("doc_or_index_changed") or signals.get("retrieval_miss_up"):
        return "retrieval-driven hallucination", "Inspect indexing/embeddings/chunks (Sample 3)"
    if signals.get("confident_but_wrong") and not signals.get("has_citations"):
        return "hallucination", "Add/repair grounding; require citations; verify retrieval"
    if signals.get("reasoning_heavy") and signals.get("small_model"):
        return "model mismatch", "Upgrade tier or add chain-of-thought"
    return "unknown", "Gather more signals; replay with logging from Phase 3"

print(diagnose({"format_broken": True}))
print(diagnose({"doc_or_index_changed": True, "confident_but_wrong": True}))
print(diagnose({"reasoning_heavy": True, "small_model": True}))

## Code-Along: Step 2 — Apply it to the Sample 3 scenario

We feed the exact Sample 3 situation — confident-but-wrong after a document refresh, model and latency unchanged — and confirm the tree points at retrieval/indexing.

In [ ]:
sample3 = {"confident_but_wrong": True, "doc_or_index_changed": True,
           "model_changed": False, "latency_changed": False, "has_citations": True}
cause, fix = diagnose(sample3)
print("Diagnosis:", cause)
print("First action:", fix)
assert "retrieval" in cause  # matches exam answer B

## Exercise

Add a branch for **cost/latency regressions** (not just quality): e.g. output tokens ballooning (runaway CoT) or retries spiking. Extend `diagnose` and add two scenarios. Then write a short note mapping each diagnosis to which Phase 3 log field would confirm it.

In [ ]:
# TODO: add cost/latency regression branches to diagnose()
# TODO: two new scenarios + the log field (from Lesson 3.4) that confirms each

## Key Takeaways

- Triage by symptom: format=prompt, unsupported=hallucination/retrieval, hard-task+small=mismatch.
- Most 'hallucinations' in RAG are really retrieval/grounding failures (Sample 3).
- A decision tree keeps diagnosis consistent and fast under pressure.
- Confirm hypotheses with the structured logs you designed in Phase 3.

---

**Next:** [Lesson 4.5 — Optimizing Token, Latency, and Cost](./lesson-4.5-optimize-token-latency-cost.ipynb)